# Neovisna evaluacija — TEyeD Dikablis

Notebook **ne trenira** modele. Kutne pogreške, off-center i statistički testovi računaju se isključivo iz spremljenih `test_predictions.csv`. Datoteke `best_model.keras` učitavaju se **samo** za mjerenje latencije. Globalni trivijalni baseline pripada glavnoj usporedbi; per-subject baseline prikazuje se zasebno kao **cheating baseline** (koristi `subject_id`, koji pravi model ne koristi). Nedostajuće datoteke se preskaču uz upozorenje; neispravna shema predikcija prekida izvođenje.

In [1]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

ROOT = Path('/home/fipulab/anastazija')
DATA = ROOT / 'TEyeD_preprocessed/128x128_stride10_q4_seg_pupil-iris-lid_2D/test'
RUNS = ROOT / 'runs'
RESULTS = ROOT / 'results'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
MTL_CONFIGS = ['mtl_full', 'mtl_wseg0', 'mtl_pupil']
MTL_SEEDS = [42, 123, 2024]
GAZE_SEEDS = [42, 123, 2024, 7, 13]
REQUIRED = ['subject_id','video_id','filename','gt_x','gt_y','gt_z','pred_x','pred_y','pred_z','angle_deg']
paths = {}
for cfg in MTL_CONFIGS:
    for seed in MTL_SEEDS:
        paths[(cfg, str(seed))] = RUNS / 'mtl' / cfg / f'seed_{seed}' / 'test_predictions.csv'
for seed in GAZE_SEEDS:
    paths[('gaze_only', str(seed))] = RUNS / 'gaze_only' / f'seed_{seed}' / 'test_predictions.csv'
for typ in ['global', 'per_subject']:
    paths[(f'trivial_baseline/{typ}', 'none')] = RUNS / 'trivial_baseline' / typ / 'test_predictions.csv'

print('Predikcijske datoteke:')
for key, path in paths.items():
    if path.is_file(): print('  PRONAĐENO:', path)
    else: warnings.warn(f'Nedostaje datoteka: {path}')
try:
    import tensorflow as tf
    devices = {'gpu': tf.config.list_physical_devices('GPU'), 'cpu': tf.config.list_physical_devices('CPU')}
    try: tf.config.experimental.set_synchronous_execution(True)
    except Exception as e: warnings.warn(f'Postavljanje sinkronog izvršavanja nije uspjelo: {e}')
except ImportError:
    tf = None
    devices = {'gpu': [], 'cpu': []}
    warnings.warn('TensorFlow nije instaliran; latencija će se preskočiti.')
print('Dostupni uređaji:', {k: [d.name for d in v] for k,v in devices.items()})

Predikcijske datoteke:
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_full/seed_42/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_full/seed_123/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_full/seed_2024/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_42/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_123/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_2024/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_42/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_123/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_2024/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/gaze_only/seed_42/test_predictions.csv
  PRONAĐENO: /home/fipulab/anastazija/runs/gaze_only/seed_123/test_predictions.csv
  PRONAĐENO: /home/fipulab/anast

2026-11-06 10:10:38.622613: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-11-06 10:10:39.132007: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/fipulab/cuda/include:/home/fipulab/cuda/lib64:/home/fipulab/cuda:/usr/lib:/usr/include
2026-11-06 10:10:39.132072: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/fipulab/cuda/include:/home/fi

Dostupni uređaji: {'gpu': ['/physical_device:GPU:0'], 'cpu': ['/physical_device:CPU:0']}


2026-11-06 10:10:39.538713: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-11-06 10:10:39.543414: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-11-06 10:10:39.543983: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [2]:
# Validacija SVIH dostupnih datoteka prije agregacije.
frames = {}
errors = []
for key, path in paths.items():
    if not path.is_file():
        continue
    df = pd.read_csv(path, dtype={'subject_id': str, 'video_id': str, 'filename': str})
    missing = [col for col in REQUIRED if col not in df.columns]
    if missing:
        errors.append(f'Nedostaju stupci u {path}: {missing}. Pronađeni: {list(df.columns)}.')
    else:
        frames[key] = df[REQUIRED].copy()
if errors:
    raise ValueError('\n'.join(errors))
if not frames:
    raise RuntimeError('Nijedan test_predictions.csv nije pronađen; evaluacija nije moguća.')
print(f'Validacija formata uspješna: {len(frames)} datoteka, svih 10 obaveznih stupaca.')

Validacija formata uspješna: 16 datoteka, svih 10 obaveznih stupaca.


In [3]:
# Agregacija kutnih pogrešaka i srednjih pogrešaka po ispitaniku.
def describe_angles(values):
    a = np.asarray(values, dtype=float)
    a = a[np.isfinite(a)]
    return dict(mean_angle=float(np.mean(a)) if len(a) else np.nan,
                std_angle=float(np.std(a, ddof=1)) if len(a)>1 else np.nan,
                median_angle=float(np.median(a)) if len(a) else np.nan,
                p95_angle=float(np.percentile(a,95)) if len(a) else np.nan)

metrics, subject_errors = {}, {}
for key, df in frames.items():
    if df['angle_deg'].isna().any() or not np.isfinite(pd.to_numeric(df['angle_deg'], errors='coerce')).all():
        warnings.warn(f'Nevaljane kutne pogreške u {paths[key]}; nevaljane vrijednosti izostavljene iz statistike.')
    vals = pd.to_numeric(df['angle_deg'], errors='coerce')
    metrics[key] = {**describe_angles(vals), 'num_subjects': int(df.subject_id.nunique()), 'num_samples': len(df)}
    subject_errors[key] = df.assign(angle_deg=vals).groupby('subject_id')['angle_deg'].mean()
print(pd.DataFrame([{'config':k[0],'seed':k[1],**v} for k,v in metrics.items()]).to_string(index=False))

                      config seed  mean_angle  std_angle  median_angle  p95_angle  num_subjects  num_samples
                    mtl_full   42    3.171311   2.173065      2.713573   7.415785            11        79964
                    mtl_full  123    3.370197   2.164627      2.926701   7.272691            11        79964
                    mtl_full 2024    3.162619   2.139615      2.786707   7.444568            11        79964
                   mtl_wseg0   42    2.957337   2.438394      2.338624   7.863587            11        79964
                   mtl_wseg0  123    3.183061   2.323063      2.600830   7.620145            11        79964
                   mtl_wseg0 2024    2.598186   1.962993      2.082247   6.406807            11        79964
                   mtl_pupil   42    3.046103   2.247889      2.586415   7.659249            11        79964
                   mtl_pupil  123    3.244702   2.236338      2.808886   7.651632            11        79964
                   

In [4]:
# Kontrola konzistentnosti kutne pogreške: točno 10 reproducibilno odabranih uzoraka.
key = ('mtl_full', '42')
if key not in frames:
    warnings.warn(f'Provjera konzistentnosti preskočena: {paths[key]}')
else:
    df = frames[key]
    np.random.seed(42)  # Samo reproducibilnost odabira uzoraka; nema treniranja.
    sample = df.sample(n=min(10,len(df)), random_state=42).copy()
    pred = sample[['pred_x','pred_y','pred_z']].to_numpy(float)
    gt = sample[['gt_x','gt_y','gt_z']].to_numpy(float)
    pred_norm = np.linalg.norm(pred, axis=1)
    gt_norm = np.linalg.norm(gt, axis=1)
    if not (np.allclose(pred_norm,1,atol=1e-5) and np.allclose(gt_norm,1,atol=1e-5)):
        warnings.warn('Pred/GT nisu jedinični vektori; za provjeru se koriste normalizirani vektori.')
    pred = pred / np.maximum(pred_norm[:,None],1e-12)
    gt = gt / np.maximum(gt_norm[:,None],1e-12)
    dots = np.einsum('ij,ij->i',pred,gt)
    cos_dist = 1-dots
    recalc = np.degrees(np.arccos(np.clip(dots,-1,1)))
    from_cos_dist = np.degrees(np.arccos(np.clip(1-cos_dist,-1,1)))
    check = pd.DataFrame({'cosine_distance':cos_dist,'angle_deg':sample.angle_deg.to_numpy(float),
                          'angle_deg_recomputed':recalc,
                          'abs_difference':np.abs(sample.angle_deg.to_numpy(float)-recalc),
                          'angle_from_cosine_distance':from_cos_dist,
                          'abs_difference_cosine':np.abs(sample.angle_deg.to_numpy(float)-from_cos_dist)})
    print(check.to_string(index=False))
    if (check.abs_difference > 0.1).any():
        warnings.warn('Nekonzistentnost kutne pogreške — mogući bug')
    print('Maksimalno odstupanje:',check.abs_difference.max(), 'stupnjeva; očekivano < 0.01°')

 cosine_distance  angle_deg  angle_deg_recomputed  abs_difference  angle_from_cosine_distance  abs_difference_cosine
        0.002317   3.901237              3.901248        0.000011                    3.901248               0.000011
        0.004224   5.268084              5.268078        0.000006                    5.268078               0.000006
        0.002737   4.239885              4.239920        0.000035                    4.239920               0.000035
        0.000093   0.781346              0.781571        0.000225                    0.781571               0.000225
        0.001953   3.581679              3.581753        0.000075                    3.581753               0.000075
        0.000708   2.156312              2.156483        0.000171                    2.156483               0.000171
        0.002208   3.808172              3.808170        0.000002                    3.808170               0.000002
        0.000539   1.881280              1.881409        0.00012

In [5]:
# Off-center: obje definicije i tri praga; per-subject cheating baseline nije u glavnoj analizi.
off_rows = []
for (cfg, seed), df in frames.items():
    if cfg == 'trivial_baseline/per_subject':
        continue
    gx = pd.to_numeric(df.gt_x, errors='coerce').to_numpy(float)
    gy = pd.to_numeric(df.gt_y, errors='coerce').to_numpy(float)
    for definition in ['axis_aligned','radial']:
        for threshold in [0.2,0.3,0.4]:
            mask = ((np.abs(gx)>threshold)|(np.abs(gy)>threshold)) if definition=='axis_aligned' else (np.hypot(gx,gy)>threshold)
            for subset, submask in [('off',mask),('on',~mask)]:
                sub = df.loc[submask]
                stat = describe_angles(pd.to_numeric(sub.angle_deg,errors='coerce'))
                nsub = sub.subject_id.nunique()
                off_rows.append({'config':cfg,'seed':seed,'definition':definition,'threshold':threshold,
                                 'subset':subset,'mean_angle':stat['mean_angle'],'std_angle':stat['std_angle'],
                                 'num_samples':len(sub),'num_subjects':int(nsub)})
                if subset=='off' and (len(sub)<100 or nsub<5):
                    warnings.warn(f'Off-center podskup je mali (n={len(sub)}, subjects={nsub}, config={cfg}, seed={seed}, def={definition}, threshold={threshold}) — mean angle error je nestabilan i treba ga tumačiti s oprezom.')
off_df = pd.DataFrame(off_rows,columns=['config','seed','definition','threshold','subset','mean_angle','std_angle','num_samples','num_subjects'])
off_path = RESULTS/'off_center_metrics.csv'
off_df.to_csv(off_path,index=False)
print('Spremljeno:',off_path)

Spremljeno: /home/fipulab/anastazija/results/off_center_metrics.csv


In [6]:
# Latencija: učitavanje modela SAMO ovdje; sinkrono izvođenje, batch=1.
# Mjerenja uključuju izvršavanje na odabranom uređaju, bez treniranja.
print("Napomena: CPU mjerenja izvode se unutar iste TF sesije s "
      "vidljivim GPU-om koristeći tf.device('/CPU:0'). Za strogo "
      "'edge CPU' mjerenje preporučuje se pokretanje s "
      "CUDA_VISIBLE_DEVICES='' u zasebnom procesu.")
latency = {'batch_size':1,'input_shape':[128,128,3],
           'measurement_note': "CPU mjerenja izvedena unutar GPU sesije preko tf.device('/CPU:0'); nije ekvivalentno čistom CPU-only procesu.",
           'models':{}}
model_paths = {'mtl_full/seed_42': RUNS/'mtl/mtl_full/seed_42/best_model.keras',
               'gaze_only/seed_42': RUNS/'gaze_only/seed_42/best_model.keras'}

def resolve_image_path(row):
    filename = str(row.filename)
    video = str(row.video_id)
    candidates = [DATA/video/filename, DATA/filename]
    for p in candidates:
        if p.is_file(): return p
    return None

def get_gaze(output):
    # Višeizlazni MTL: gaze izlaz može biti dict ili lista.
    if isinstance(output, dict):
        keys = [k for k in output if 'gaze' in k.lower()]
        if not keys: raise ValueError(f'Nije moguće identificirati gaze izlaz: {list(output)}')
        return output[keys[0]]
    if isinstance(output,(list,tuple)):
        hits = [v for v in output if len(v.shape)==2 and v.shape[-1]==3]
        if len(hits)!=1: raise ValueError('Nije moguće jednoznačno identificirati gaze izlaz modela.')
        return hits[0]
    return output

def preprocess_image(path):
    raw = tf.io.read_file(str(path))
    image = tf.io.decode_jpeg(raw, channels=3)
    image = tf.image.resize(image,[128,128])
    return tf.expand_dims(tf.cast(image,tf.float32)/127.5-1.0,axis=0)

def _sync(result):
    # Sinkronizacija svih izlaznih tenzora prije završetka mjerenja.
    if isinstance(result, dict):
        for value in result.values():
            if hasattr(value, 'numpy'):
                _ = value.numpy()
    elif isinstance(result, (list, tuple)):
        for value in result:
            _sync(value)
    elif hasattr(result, 'numpy'):
        _ = result.numpy()

def measure(fn,warmup,repeats):
    for _ in range(warmup):
        result = fn()
        _sync(result)
    times=[]
    for _ in range(repeats):
        start=time.perf_counter()
        result=fn()
        _sync(result)
        times.append((time.perf_counter()-start)*1000)
    a=np.asarray(times)
    med=float(np.median(a))
    return {'mean_ms':float(a.mean()),'std_ms':float(a.std(ddof=1)),
            'median_ms':med,'p95_ms':float(np.percentile(a,95)),
            'fps':float(1000/med) if med>0 else np.nan,'n_measurements':repeats}

if tf is None:
    warnings.warn('Latencija preskočena: TensorFlow nije dostupan.')
else:
    # Ako je jedna datoteka nedostupna, druga se ipak mjeri.
    for model_name, model_path in model_paths.items():
        latency['models'][model_name]={}
        if not model_path.is_file():
            warnings.warn(f'Nedostaje model za latenciju: {model_path}')
            continue
        if not len(frames):
            warnings.warn('Nema predikcija za pronalazak test slike.')
            continue
        image_path = None
        for df in frames.values():
            if len(df):
                image_path=resolve_image_path(df.iloc[0])
                if image_path is not None: break
        if image_path is None:
            warnings.warn(f'Ne postoji dostupna test slika za latenciju modela {model_name}.')
            continue
        try:
            model = tf.keras.models.load_model(str(model_path),compile=False)
        except Exception as e:
            warnings.warn(f'Neuspjelo učitavanje {model_path}: {e}')
            continue
        # Dijagnostika smještaja težina učitanog modela.
        try:
            first_layer = next(layer for layer in model.layers if layer.weights)
            print(f'  Model device (prvi sloj s težinama): {first_layer.weights[0].device}')
        except StopIteration:
            print('  Model device: nije moguće odrediti (nema slojeva s težinama)')
        for device_name, physical in devices.items():
            if not physical:
                warnings.warn(f'Uređaj {device_name.upper()} nije dostupan; preskačem mjerenje.')
                continue
            device='/GPU:0' if device_name=='gpu' else '/CPU:0'
            try:
                with tf.device(device):
                    fixed=preprocess_image(image_path)
                    # Graf se kompilira i trasira prije warmupa, izvan mjerenja.
                    @tf.function(input_signature=[tf.TensorSpec(shape=[1,128,128,3], dtype=tf.float32)])
                    def forward_compiled(images):
                        return model(images, training=False)

                    _ = forward_compiled(fixed)

                    @tf.function(input_signature=[tf.TensorSpec(shape=[1,128,128,3], dtype=tf.float32)])
                    def forward_for_e2e(images):
                        return model(images, training=False)

                    _ = forward_for_e2e(preprocess_image(image_path))

                    def forward():
                        return forward_compiled(fixed)

                    def end_to_end():
                        image=preprocess_image(image_path)
                        gaze=get_gaze(forward_for_e2e(image))
                        gaze=tf.math.l2_normalize(tf.cast(gaze,tf.float32),axis=-1)
                        return gaze
                    latency['models'][model_name][f'{device_name}_model_only']=measure(forward,100,1000 if device_name=='gpu' else 200)
                    latency['models'][model_name][f'{device_name}_end_to_end']=measure(end_to_end,20,200 if device_name=='gpu' else 100)
                    print(model_name,device_name,latency['models'][model_name])
            except Exception as e:
                warnings.warn(f'Mjerenje latencije neuspjelo ({model_name}, {device_name}): {e}')
        del model
# Rekurzivna sanitizacija nevaljanih numeričkih vrijednosti za strogi JSON.
def sanitize_for_json(obj, path='latency'):
    if isinstance(obj, dict):
        return {k: sanitize_for_json(v, f'{path}.{k}') for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [sanitize_for_json(v, f'{path}[{i}]') for i, v in enumerate(obj)]
    if isinstance(obj, (float, np.floating)):
        if np.isnan(obj):
            warnings.warn(f'NaN vrijednost u latency rezultatima zamijenjena s null: {path}')
            return None
        if not np.isfinite(obj):
            warnings.warn(f'Beskonačna vrijednost u latency rezultatima zamijenjena s null: {path}')
            return None
        return float(obj)
    return obj

latency_path=RESULTS/'latency.json'
sanitized_latency=sanitize_for_json(latency)
latency_path.write_text(json.dumps(sanitized_latency,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('Spremljeno:',latency_path)

Napomena: CPU mjerenja izvode se unutar iste TF sesije s vidljivim GPU-om koristeći tf.device('/CPU:0'). Za strogo 'edge CPU' mjerenje preporučuje se pokretanje s CUDA_VISIBLE_DEVICES='' u zasebnom procesu.
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3090, compute capability 8.6


2026-11-06 10:10:41.706364: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-11-06 10:10:41.789458: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-11-06 10:10:41.791080: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-11-06 10:10:41.791680: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least on

  Model device (prvi sloj s težinama): /job:localhost/replica:0/task:0/device:GPU:0


2026-11-06 10:10:43.995972: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8100
2026-11-06 10:10:44.939523: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:630] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


mtl_full/seed_42 gpu {'gpu_model_only': {'mean_ms': 12.277126512723044, 'std_ms': 0.8383013983722527, 'median_ms': 12.832261039875448, 'p95_ms': 13.026898528914899, 'fps': 77.9285892714123, 'n_measurements': 1000}, 'gpu_end_to_end': {'mean_ms': 13.534577505779453, 'std_ms': 0.29890587231151455, 'median_ms': 13.502160436473787, 'p95_ms': 14.385472232243044, 'fps': 74.06222172406353, 'n_measurements': 200}}
mtl_full/seed_42 cpu {'gpu_model_only': {'mean_ms': 12.277126512723044, 'std_ms': 0.8383013983722527, 'median_ms': 12.832261039875448, 'p95_ms': 13.026898528914899, 'fps': 77.9285892714123, 'n_measurements': 1000}, 'gpu_end_to_end': {'mean_ms': 13.534577505779453, 'std_ms': 0.29890587231151455, 'median_ms': 13.502160436473787, 'p95_ms': 14.385472232243044, 'fps': 74.06222172406353, 'n_measurements': 200}, 'cpu_model_only': {'mean_ms': 143.4136725915596, 'std_ms': 14.758434917737778, 'median_ms': 148.23716098908335, 'p95_ms': 163.42741427361034, 'fps': 6.745946787753464, 'n_measurement

In [7]:
# Upareni Wilcoxon i Cohenov d po ispitaniku.
def interpret_cohens_d(d):
    # Interpretacija veličine učinka prema apsolutnoj vrijednosti Cohenova d.
    if pd.isna(d): return 'nepoznato'
    if abs(d)<0.2: return 'mala'
    if abs(d)<0.5: return 'srednja'
    return 'velika'

def paired_test_series(a_series,b_series,comparison,seed,level):
    a,b=a_series.align(b_series,join='inner')
    good=np.isfinite(a.to_numpy(float)) & np.isfinite(b.to_numpy(float))
    diff=a.to_numpy(float)[good]-b.to_numpy(float)[good]
    n=len(diff)
    if n<2:
        warnings.warn(f'Premalo uparenih ispitanika za {comparison}, seed={seed}: n={n}')
        statistic=pval=d=np.nan
    else:
        if np.all(diff==0):
            warnings.warn(f'Sve uparene razlike su nula: {comparison}, seed={seed}')
            statistic,pval=0.0,1.0
        else:
            try: statistic,pval=wilcoxon(diff,zero_method='wilcox')
            except ValueError as e:
                warnings.warn(f'Wilcoxon nije uspio za {comparison}: {e}')
                statistic=pval=np.nan
        sd=diff.std(ddof=1)
        d=float(diff.mean()/sd) if sd>0 else np.nan
    return {'comparison':comparison,'seed':seed,'statistic':float(statistic),'p_value':float(pval),
            'n_subjects':n,'cohens_d':float(d),
            'cohens_d_interpretation':interpret_cohens_d(d),'level':level}

def paired_test(key_a,key_b,comparison,seed,level):
    # Wrapper zadržava postojeći način pozivanja za pojedinačne seedove.
    return paired_test_series(subject_errors[key_a],subject_errors[key_b],comparison,seed,level)

def pooled_subject(keys):
    series=[subject_errors[k].rename(f'seed_{k[1]}') for k in keys if k in subject_errors]
    return pd.concat(series,axis=1).mean(axis=1) if series else None

def pooled_test(keys_a,keys_b,comparison):
    a,b=pooled_subject(keys_a),pooled_subject(keys_b)
    if a is None or b is None: return None
    return paired_test_series(a,b,comparison,'pooled_subject','across_seeds')

# Najbolji MTL bira se isključivo unutar mtl_full; ostale konfiguracije su ablacije.
mtl_full_keys=[k for k in metrics if k[0]=='mtl_full' and np.isfinite(metrics[k]['mean_angle'])]
gaze_keys=[k for k in metrics if k[0]=='gaze_only' and np.isfinite(metrics[k]['mean_angle'])]
best_mtl=min(mtl_full_keys,key=lambda k:metrics[k]['mean_angle']) if mtl_full_keys else None
best_gaze=min(gaze_keys,key=lambda k:metrics[k]['mean_angle']) if gaze_keys else None
global_key=('trivial_baseline/global','none')
base_rows=[]
for a,b,name in [(best_mtl,global_key,'best_mtl_vs_trivial_global'),
                 (best_gaze,global_key,'gaze_only_vs_trivial_global'),
                 (best_mtl,best_gaze,'best_mtl_vs_gaze_only')]:
    if a in subject_errors and b in subject_errors:
        base_rows.append(paired_test(a,b,name,a[1],'per_seed'))
    else: warnings.warn(f'Preskačem usporedbu {name}: nedostaju podaci.')
base_wilcoxon=pd.DataFrame(base_rows,columns=['comparison','seed','statistic','p_value','n_subjects','cohens_d','cohens_d_interpretation','level'])
base_wilcoxon_path=RESULTS/'wilcoxon_vs_baseline.csv'
base_wilcoxon.to_csv(base_wilcoxon_path,index=False)
print('Najbolji MTL:',best_mtl,'; najbolji gaze-only:',best_gaze)
print(base_wilcoxon.to_string(index=False))

Najbolji MTL: ('mtl_full', '2024') ; najbolji gaze-only: ('gaze_only', '7')
                 comparison seed  statistic  p_value  n_subjects  cohens_d cohens_d_interpretation    level
 best_mtl_vs_trivial_global 2024        4.0 0.006836          11 -1.187134                  velika per_seed
gaze_only_vs_trivial_global    7       20.0 0.278320          11  0.349925                 srednja per_seed
      best_mtl_vs_gaze_only 2024        4.0 0.006836          11 -1.236506                  velika per_seed


In [8]:
# Neovisna provjera MTL vs gaze-only po istim seedovima te pooled po ispitaniku.
wilcoxon_rows=[]
for cfg in MTL_CONFIGS:
    matched=[]
    for seed in MTL_SEEDS:
        a,b=(cfg,str(seed)),('gaze_only',str(seed))
        if a in subject_errors and b in subject_errors:
            wilcoxon_rows.append(paired_test(a,b,f'{cfg}_vs_gaze_only',str(seed),'per_seed'))
            matched.append(seed)
        else:
            warnings.warn(f'Nedostaje upareni seed: {cfg}, gaze_only, seed={seed}')
    if matched:
        row=pooled_test([(cfg,str(s)) for s in matched], [('gaze_only',str(s)) for s in matched], f'{cfg}_vs_gaze_only')
        if row: wilcoxon_rows.append(row)
wilcoxon_df=pd.DataFrame(wilcoxon_rows,columns=['comparison','seed','statistic','p_value','n_subjects','cohens_d','cohens_d_interpretation','level'])
wilcoxon_path=RESULTS/'wilcoxon.csv'
wilcoxon_df.to_csv(wilcoxon_path,index=False)

# Usporedba s izvornim PROMPT 3 rezultatima samo za usporedive parove i seedove.
original_path=RUNS/'mtl/wilcoxon.csv'
if original_path.is_file():
    original=pd.read_csv(original_path,dtype={'seed':str})
    if {'comparison','seed','statistic','p_value'}.issubset(original.columns):
        current=pd.concat([wilcoxon_df,base_wilcoxon],ignore_index=True)
        current['seed']=current['seed'].astype(str)
        original['seed']=original['seed'].astype(str)
        both=current.merge(original,on=['comparison','seed'],suffixes=('_new','_original'))
        unmatched=current.merge(original[['comparison','seed','statistic']],
                                on=['comparison','seed'],how='left',suffixes=('', '_original'))
        unmatched_count=int(unmatched['statistic_original'].isna().sum())
        print(f'Wilcoxon usporedba s PROMPT 3: uspoređeno parova={len(both)}, '
              f'bez para u izvorniku={unmatched_count}')
        if len(both)==0:
            warnings.warn("Nije pronađen nijedan zajednički (comparison, seed) par "
                          "između novog i PROMPT 3 Wilcoxona — provjeri nazive u "
                          "stupcu 'comparison' i format 'seed'. Usporedba je preskočena.")
        for col in ['statistic','p_value']:
            a=pd.to_numeric(both[f'{col}_new'],errors='coerce')
            b=pd.to_numeric(both[f'{col}_original'],errors='coerce')
            if ((a-b).abs()>1e-6).any():
                warnings.warn('Nekonzistentnost s PROMPT 3 Wilcoxon rezultatima')
    else:
        warnings.warn(f'Nije moguće usporediti format izvornog Wilcoxona: {original_path}')
else:
    warnings.warn(f'Nedostaje datoteka za usporedbu Wilcoxona: {original_path}')
print(wilcoxon_df.to_string(index=False))

Wilcoxon usporedba s PROMPT 3: uspoređeno parova=0, bez para u izvorniku=15
            comparison           seed  statistic  p_value  n_subjects  cohens_d cohens_d_interpretation        level
 mtl_full_vs_gaze_only             42        3.0 0.004883          11 -1.427460                  velika     per_seed
 mtl_full_vs_gaze_only            123        1.0 0.001953          11 -1.607862                  velika     per_seed
 mtl_full_vs_gaze_only           2024        4.0 0.006836          11 -1.326968                  velika     per_seed
 mtl_full_vs_gaze_only pooled_subject        3.0 0.004883          11 -1.468899                  velika across_seeds
mtl_wseg0_vs_gaze_only             42        4.0 0.006836          11 -1.242683                  velika     per_seed
mtl_wseg0_vs_gaze_only            123        2.0 0.002930          11 -1.535457                  velika     per_seed
mtl_wseg0_vs_gaze_only           2024        1.0 0.001953          11 -1.801555                  velika  

/tmp/ipykernel_388729/2817298905.py:34: UserWarning: Nije pronađen nijedan zajednički (comparison, seed) par između novog i PROMPT 3 Wilcoxona — provjeri nazive u stupcu 'comparison' i format 'seed'. Usporedba je preskočena.
  warnings.warn("Nije pronađen nijedan zajednički (comparison, seed) par "


In [9]:
# Završni sažetak; per-subject se izdvaja iz glavne Markdown tablice.
def off_value(key,definition,subset):
    cfg,seed=key
    sub=off_df[(off_df.config==cfg)&(off_df.seed==seed)&
               (off_df.definition==definition)&(off_df.threshold==0.3)&(off_df.subset==subset)]
    return float(sub.iloc[0].mean_angle) if len(sub) else np.nan

rows=[]
for key, stat in metrics.items():
    cfg,seed=key
    latency_key=f'{cfg}/seed_{seed}'
    lt=latency['models'].get(latency_key,{})
    rows.append({'config':cfg if cfg!='trivial_baseline/per_subject' else 'trivial_baseline/per_subject (cheating)',
                 'seed':seed,**stat,
                 'off_center_error_axis':off_value(key,'axis_aligned','off'),
                 'off_center_error_radial':off_value(key,'radial','off'),
                 'on_center_error_axis':off_value(key,'axis_aligned','on'),
                 'fps_gpu_model_only':lt.get('gpu_model_only',{}).get('fps',np.nan),
                 'fps_gpu_end_to_end':lt.get('gpu_end_to_end',{}).get('fps',np.nan)})
summary_columns=['config','seed','mean_angle','std_angle','median_angle','p95_angle',
                 'off_center_error_axis','off_center_error_radial','on_center_error_axis',
                 'fps_gpu_model_only','fps_gpu_end_to_end','num_subjects','num_samples']
final_df=pd.DataFrame(rows,columns=summary_columns)
summary_path=RESULTS/'final_summary.csv'
final_df.to_csv(summary_path,index=False)
across=(final_df[~final_df.config.str.contains('trivial_baseline')]
        .groupby('config').mean_angle.agg(['mean','std']).reset_index()
        .rename(columns={'mean':'mean_angle_mean_across_seeds','std':'mean_angle_std_across_seeds'}))
print('Sažetak preko seedova:')
print(across.to_string(index=False))

def fmt(value,precision=2):
    return f'{value:.{precision}f}' if pd.notna(value) and np.isfinite(value) else '—'

def md_table(data):
    headers=['Konfiguracija','Seed','Kutna pogreška (°)','Medijan (°)','P95 (°)',
             'Off-center osno (°)','Off-center radijalno (°)','On-center osno (°)',
             'FPS GPU model','FPS GPU ukupno','Ispitanici','Uzorci']
    lines=['| '+' | '.join(headers)+' |','| '+' | '.join(['---']*len(headers))+' |']
    for _,r in data.iterrows():
        vals=[r.config,r.seed,f'{fmt(r.mean_angle)} ± {fmt(r.std_angle)}',fmt(r.median_angle),fmt(r.p95_angle),
              fmt(r.off_center_error_axis),fmt(r.off_center_error_radial),fmt(r.on_center_error_axis),
              fmt(r.fps_gpu_model_only,1),fmt(r.fps_gpu_end_to_end,1),str(r.num_subjects),str(r.num_samples)]
        lines.append('| '+' | '.join(map(str,vals))+' |')
    return '\n'.join(lines)

main=final_df[~final_df.config.str.contains('per_subject')]
cheating=final_df[final_df.config.str.contains('per_subject')]
md_lines=['# Rezultati neovisne evaluacije','',
          'Kutne pogreške izražene su u stupnjevima. Off-center definicije koriste prag 0,3.', '',
          '## Glavna usporedba','',md_table(main),'',
          '## Prosjek preko seedova','',
          '| Konfiguracija | Srednja kutna pogreška preko seedova (°) |',
          '| --- | --- |']
for _,r in across.iterrows():
    md_lines.append(f'| {r.config} | {fmt(r.mean_angle_mean_across_seeds)} ± {fmt(r.mean_angle_std_across_seeds)} |')
md_lines.extend([
    '',
    '## Per-subject trivijalni baseline',
    '',
    '**Napomena:** Per-subject baseline je identičan globalnom '
    '(6,18°), jer su svi testni ispitanici subject-disjoint — model '
    'nema per-subject povijest za test ispitanike i koristi globalni '
    'fallback. Nije zasebno informativan i ne prijavljuje se kao '
    'zaseban rezultat u glavnoj usporedbi.',
    '',
    'U final_summary.csv per-subject baseline je prisutan radi '
    'potpunosti (označen kao "cheating"), ali brojke su identične '
    'globalnom baselineu.'
])
md_path=RESULTS/'final_summary.md'
md_path.write_text('\n'.join(md_lines)+'\n',encoding='utf-8')
print('Spremljeno:',summary_path,md_path)

Sažetak preko seedova:
   config  mean_angle_mean_across_seeds  mean_angle_std_across_seeds
gaze_only                      6.447131                     0.084402
 mtl_full                      3.234709                     0.117416
mtl_pupil                      3.077456                     0.153981
mtl_wseg0                      2.912861                     0.294963
Spremljeno: /home/fipulab/anastazija/results/final_summary.csv /home/fipulab/anastazija/results/final_summary.md


In [10]:
# Vizualizacije: boxplot prosjeka po ispitaniku preko seedova, histogram i 2D scatter.
figure_paths=[]
plot_cfgs=['mtl_full','mtl_pupil','gaze_only']
series=[]
for cfg in plot_cfgs:
    keys=[k for k in subject_errors if k[0]==cfg]
    pooled=pooled_subject(keys)
    if pooled is not None and len(pooled.dropna()):
        series.append((cfg,pooled.dropna().to_numpy(float)))
    else: warnings.warn(f'Nema podataka za boxplot: {cfg}')
if series:
    fig,ax=plt.subplots(figsize=(8,5))
    ax.boxplot([v for _,v in series],labels=[k for k,_ in series])
    ax.set(ylabel='Srednja kutna pogreška po ispitaniku (°)',title='Kutna pogreška po ispitaniku — top 3 modela')
    fig.tight_layout()
    p=FIGURES/'subject_boxplot.png';fig.savefig(p,dpi=180);plt.close(fig);figure_paths.append(p)

full_keys=[k for k in metrics if k[0]=='mtl_full' and np.isfinite(metrics[k]['mean_angle'])]
if full_keys:
    best_full=min(full_keys,key=lambda k:metrics[k]['mean_angle'])
    df=frames[best_full]
    angles=pd.to_numeric(df.angle_deg,errors='coerce').dropna()
    fig,ax=plt.subplots(figsize=(8,5))
    ax.hist(angles,bins=50)
    ax.axvline(angles.median(),linestyle='--',label=f'Medijan {angles.median():.2f}°')
    ax.axvline(angles.mean(),linestyle=':',label=f'Sredina {angles.mean():.2f}°')
    ax.set(xlabel='Kutna pogreška (°)',ylabel='Broj uzoraka',title=f'MTL full — seed {best_full[1]}')
    ax.legend();fig.tight_layout()
    p=FIGURES/'mtl_full_histogram.png';fig.savefig(p,dpi=180);plt.close(fig);figure_paths.append(p)

    fig,ax=plt.subplots(figsize=(7,7))
    ax.scatter(df.gt_x,df.gt_y,s=5,alpha=.3,c='blue',label='GT')
    ax.scatter(df.pred_x,df.pred_y,s=5,alpha=.3,c='red',label='Pred')
    # Dijagonala y=x prikazuje se kao referenca koordinatnog prostora, ne kao regresijski fit.
    coords=pd.concat([df[['gt_x','gt_y','pred_x','pred_y']].stack()],ignore_index=True)
    low,high=float(coords.min()),float(coords.max())
    ax.plot([low,high],[low,high],color='gray',linestyle='--',linewidth=1,label='y=x')
    ax.set(xlabel='Gaze x',ylabel='Gaze y',title=f'Predikcija vs GT — 2D projekcija (x, y), mtl_full (seed {best_full[1]})')
    ax.set_aspect('equal',adjustable='box');ax.legend();fig.tight_layout()
    p=FIGURES/'pred_vs_gt_2d.png';fig.savefig(p,dpi=180);plt.close(fig);figure_paths.append(p)
else:
    warnings.warn('Nedostaje mtl_full: histogram i scatter nisu generirani.')
print('Generirane slike:',*[str(p) for p in figure_paths],sep='\n  ')

Generirane slike:
  /home/fipulab/anastazija/results/figures/subject_boxplot.png
  /home/fipulab/anastazija/results/figures/mtl_full_histogram.png
  /home/fipulab/anastazija/results/figures/pred_vs_gt_2d.png


In [11]:
# Završni izvještaj s putanjama i ključnim brojkama.
print('\n=== ZAVRŠETAK EVALUACIJE ===')
for label,key in [('Najbolji MTL (unutar mtl_full)',best_mtl),('Najbolji gaze-only',best_gaze),
                  ('Globalni trivijalni baseline',global_key),
                  ('Per-subject cheating baseline (identičan globalnom — subject-disjoint)',('trivial_baseline/per_subject','none'))]:
    if key in metrics:
        v=metrics[key]
        print(f'{label} ({key[0]}, seed={key[1]}): {v["mean_angle"]:.2f} ± {v["std_angle"]:.2f}°')
    else: print(f'{label}: nije dostupan')
print('\nGenerirane datoteke:')
for path in [off_path,latency_path,base_wilcoxon_path,wilcoxon_path,summary_path,md_path,*figure_paths]:
    print(' ',path)
print('\nNapomena: per-subject baseline je cheating baseline; koristi subject_id, za razliku od stvarnih modela.')
print('\nNapomena: per-subject baseline je identičan globalnom '
      '(subject-disjoint podjela); nije zasebno informativan.')
print('Napomena: mtl_wseg0 i mtl_pupil su ablacije — prikazani u '
      'final_summary.csv, ali nisu "najbolji MTL" model.')
# Interpretacija veličine učinka za glavne usporedbe.
for _, row in base_wilcoxon.iterrows():
    print(f"{row['comparison']}: d={row['cohens_d']:.2f} ({row['cohens_d_interpretation']})")



=== ZAVRŠETAK EVALUACIJE ===
Najbolji MTL (unutar mtl_full) (mtl_full, seed=2024): 3.16 ± 2.14°
Najbolji gaze-only (gaze_only, seed=7): 6.30 ± 4.85°
Globalni trivijalni baseline (trivial_baseline/global, seed=none): 6.18 ± 4.81°
Per-subject cheating baseline (identičan globalnom — subject-disjoint) (trivial_baseline/per_subject, seed=none): 6.18 ± 4.81°

Generirane datoteke:
  /home/fipulab/anastazija/results/off_center_metrics.csv
  /home/fipulab/anastazija/results/latency.json
  /home/fipulab/anastazija/results/wilcoxon_vs_baseline.csv
  /home/fipulab/anastazija/results/wilcoxon.csv
  /home/fipulab/anastazija/results/final_summary.csv
  /home/fipulab/anastazija/results/final_summary.md
  /home/fipulab/anastazija/results/figures/subject_boxplot.png
  /home/fipulab/anastazija/results/figures/mtl_full_histogram.png
  /home/fipulab/anastazija/results/figures/pred_vs_gt_2d.png

Napomena: per-subject baseline je cheating baseline; koristi subject_id, za razliku od stvarnih modela.

Napome